# GamaX1 / Aetherion — Final Consolidated Colab

This notebook is the execution guide for the rechecked v7 codebase. It keeps the research foundation unchanged.

**Two checkpoints are deliberately different:**
- **500 files** → corpus encoding checkpoint + timing/ETA.
- **500 optimizer steps** → model checkpoint + timing/throughput.

The notebook does not claim that the Aetherion mechanism replaces attention: current GamaX1 retains causal self-attention and replaces the FFN path with the sparse-superposition mechanism.

## 1. GPU + Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone/verify the project

In [ ]:
%cd /content
import os, subprocess
if not os.path.exists("GamaX1_Aetherion"):
    !git clone https://github.com/mrroy-dev/GamaX1_Aetherion.git
%cd /content/GamaX1_Aetherion
!python -m py_compile gamax1/*.py compare_dense.py prepare_large_corpus.py
!python - <<'PY'
from gamax1.layers import DynamicSparsityController
from gamax1.tokenizer import BPETokenizer
from gamax1.experiment_tracker import ExperimentTracker
print("imports: OK")
c=DynamicSparsityController(16,2,16); c.k=999; c.step(1.0); assert c.k==16
print("sparsity bounds: OK")
PY

## 3. Corpus configuration

In [ ]:
DATA_ROOT="/content/drive/MyDrive/Aetherion_GamaX1/data"
BULK_CACHE="/content/drive/MyDrive/Aetherion_GamaX1/cache/bulk_bpe"
CKPT_DIR="/content/drive/MyDrive/Aetherion_GamaX1/checkpoints/gamax1_final_v7"
EXPERIMENT_DIR="/content/drive/MyDrive/Aetherion_GamaX1/experiments"
print(DATA_ROOT, BULK_CACHE, CKPT_DIR, EXPERIMENT_DIR)

## 4. Read-only cache/timing inspection

In [ ]:
from pathlib import Path
import json
cache=Path(BULK_CACHE)
for name in ["tokenizer.json","tokens.int32.bin","file_index.json","metadata.json","encode_progress.json","encoding_checkpoint_timing.jsonl"]:
    p=cache/name
    print(name, "PRESENT" if p.exists() else "MISSING")
if (cache/"tokens.int32.bin").exists():
    b=(cache/"tokens.int32.bin").stat().st_size
    print("bytes:", b, "tokens:", b//4, "alignment:", b%4==0)
if (cache/"encoding_checkpoint_timing.jsonl").exists():
    print((cache/"encoding_checkpoint_timing.jsonl").read_text().splitlines()[-3:])

## 5. Real-data smoke test (recommended before a long run)

In [ ]:
!python -m gamax1.train --tokenizer bpe --data_dir "$DATA_ROOT" --bulk_cache_dir "$BULK_CACHE" --out_dir "$CKPT_DIR/smoke" --d_model 64 --n_heads 2 --n_layers 2 --n_features 256 --block_size 128 --batch_size 4 --max_steps 10 --eval_interval 5 --checkpoint_interval 5 --experiment_dir "$EXPERIMENT_DIR" --run_name smoke_v7

## 6. Full/resume training — 500-step checkpoints

In [ ]:
# IMPORTANT: run without --rebuild_bulk_cache after the first successful corpus build.
# The code itself defaults checkpoint_interval=500. It also persists per-run metrics.
TRAIN_CMD = (
    f'python -m gamax1.train --tokenizer bpe --data_dir "{DATA_ROOT}" '
    f'--bulk_cache_dir "{BULK_CACHE}" --out_dir "{CKPT_DIR}" '
    f'--d_model 768 --n_heads 12 --n_layers 12 --n_features 3072 '
    f'--block_size 512 --batch_size 16 --bpe_vocab_size 16000 '
    f'--max_steps 30000 --checkpoint_interval 500 '
    f'--experiment_dir "{EXPERIMENT_DIR}" --run_name full_v7'
)
print(TRAIN_CMD)
# Uncomment after the smoke test:
# !{TRAIN_CMD}

## 7. Inspect measured checkpoint timing / plots

In [ ]:
from pathlib import Path
import json
base=Path(EXPERIMENT_DIR)
for d in sorted([p for p in base.iterdir() if p.is_dir()])[-5:]:
    print("\nRUN",d.name)
    tp=d/"checkpoint_timing.jsonl"
    if tp.exists(): print(tp.read_text().splitlines()[-5:])
    print("plots:",[p.name for p in (d/"plots").glob("*.png")] if (d/"plots").exists() else [])

## 8. Generation / chat

In [ ]:
!python -m gamax1.generate --ckpt "$CKPT_DIR/gamax1_latest.pt" --chat --prompt "Explain what a language model does." --max_new_tokens 200 --temperature 0.8 --top_k 40 --repetition_penalty 1.2 --stop_at_eos

## 9. Sparse vs dense controlled experiment

In [ ]:
!python compare_dense.py --steps 300 --d_model 64 --n_heads 2 --n_layers 2 --n_features 256 --block_size 64 --batch_size 16 --lr 3e-4 --dropout 0 --eval_batches 20

## 10. Math/mechanism audit

In [ ]:
!python -m gamax1.mechanism_audit
!cat MATH_CORRECTION_LOG.md

## 11. Important interpretation rules

- `k / n_features` is a **local sparse-work proxy**, not a guaranteed GPU speedup.
- Wall-clock speed, tokens/sec, checkpoint interval, and memory must be measured.
- Lower validation loss/PPL is descriptive evidence; do not turn one run into a superiority claim.
- Base pretraining and instruction/SFT remain separate stages.
- Any architectural/formula correction must be made for correctness first, then experiments rerun.